## Install and imports

In [ ]:
try:
    import langgraph
    import langchain_google_genai
except ImportError:
    %pip install -q langchain-google-genai langgraph

In [ ]:
import os
import re
import sqlite3
import time
import random
from datetime import date, timedelta
from typing import TypedDict, Optional

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from langgraph.graph import StateGraph, START, END

DB_PATH = "ecommerce.db"
MODEL_NAME = "gemini-3.8-flash"   
MAX_RETRIES = 3                  

## Connect to Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except ImportError:
    api_key = os.environ.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(model=MODEL_NAME, google_api_key=api_key, temperature=0)

In [ ]:
def get_text(content):
    """Gemini can return a list of parts instead of a plain string. This always gives us the text."""
    if isinstance(content, str):
        return content
    return "".join(part.get("text", "") for part in content if isinstance(part, dict))


# Test the connection
print(get_text(llm.invoke("Reply with: Gemini is working!").content))

## Create a mock e-commerce database

In [ ]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city TEXT NOT NULL
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

customers = [
    (1, "Alice", "Cairo"), (2, "Omar", "Alexandria"), (3, "Sara", "Giza"), (4, "Youssef", "Cairo"),
    (5, "Mariam", "Alexandria"), (6, "Hassan", "Mansoura"), (7, "Nour", "Giza"), (8, "Karim", "Cairo"),
]
products = [
    (1, "Laptop", "Electronics", 25000.0), (2, "Headphones", "Electronics", 1500.0),
    (3, "Office Chair", "Furniture", 4000.0), (4, "Desk", "Furniture", 6500.0),
    (5, "Keyboard", "Accessories", 800.0), (6, "Monitor", "Electronics", 7000.0),
    (7, "Mouse", "Accessories", 400.0), (8, "Webcam", "Accessories", 1200.0),
]
cursor.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
cursor.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", products)

random.seed(42)
today = date.today()
orders, items = [], []
order_id, item_id = 1, 1

for days_ago in range(45):
    order_date = (today - timedelta(days=days_ago)).isoformat()
    for k in range(random.randint(2, 4)):
        status = "completed" if k == 0 else random.choices(
            ["completed", "pending", "cancelled"], weights=[70, 20, 10])[0]
        orders.append((order_id, random.randint(1, len(customers)), order_date, status))
        for product_id in random.sample(range(1, len(products) + 1), random.randint(1, 3)):
            items.append((item_id, order_id, product_id, random.randint(1, 3), products[product_id - 1][3]))
            item_id += 1
        order_id += 1

cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", orders)
cursor.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?, ?)", items)
conn.commit()
conn.close()

print("Customers:", len(customers), "| Products:", len(products),
      "| Orders:", len(orders), "| Order items:", len(items))

## A read-only SQL tool 


In [ ]:
readonly_uri = f"file:{DB_PATH}?mode=ro"

WRITE_KEYWORDS = re.compile(
    r"\b(INSERT|UPDATE|DELETE|REPLACE|CREATE|DROP|ALTER|ATTACH|DETACH|PRAGMA|"
    r"VACUUM|REINDEX|ANALYZE|BEGIN|COMMIT|ROLLBACK|SAVEPOINT|RELEASE)\b",
    re.IGNORECASE,
)

_denied_names = [
    "SQLITE_INSERT", "SQLITE_UPDATE", "SQLITE_DELETE", "SQLITE_CREATE_INDEX", "SQLITE_CREATE_TABLE",
    "SQLITE_CREATE_TEMP_INDEX", "SQLITE_CREATE_TEMP_TABLE", "SQLITE_CREATE_TEMP_TRIGGER",
    "SQLITE_CREATE_TEMP_VIEW", "SQLITE_CREATE_TRIGGER", "SQLITE_CREATE_VIEW", "SQLITE_DROP_INDEX",
    "SQLITE_DROP_TABLE", "SQLITE_DROP_TEMP_INDEX", "SQLITE_DROP_TEMP_TABLE", "SQLITE_DROP_TEMP_TRIGGER",
    "SQLITE_DROP_TEMP_VIEW", "SQLITE_DROP_TRIGGER", "SQLITE_DROP_VIEW", "SQLITE_ALTER_TABLE",
    "SQLITE_ATTACH", "SQLITE_DETACH", "SQLITE_PRAGMA",
]
DENIED_ACTIONS = {getattr(sqlite3, n) for n in _denied_names if hasattr(sqlite3, n)}


def authorize_read_only(action, arg1, arg2, database, source):
    return sqlite3.SQLITE_DENY if action in DENIED_ACTIONS else sqlite3.SQLITE_OK


def execute_sql(sql, timeout_seconds=5, max_rows=100):
    sql = sql.strip()
    if sql.endswith(";"):
        sql = sql[:-1].strip()

    if not sql:
        raise ValueError("SQL query is empty.")
    if ";" in sql:
        raise ValueError("Only one SQL statement is allowed.")
    if not re.match(r"^(SELECT|WITH)\b", sql, re.IGNORECASE):
        raise ValueError("Only SELECT or WITH queries are allowed.")
    if WRITE_KEYWORDS.search(sql):
        raise ValueError("Potentially unsafe SQL keyword detected.")

    connection = sqlite3.connect(readonly_uri, uri=True, timeout=5)
    try:
        connection.set_authorizer(authorize_read_only)
        deadline = time.time() + timeout_seconds
        connection.set_progress_handler(lambda: 1 if time.time() > deadline else 0, 10000)
        return pd.read_sql_query(sql, connection).head(max_rows)
    finally:
        connection.close()


display(execute_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"))

## Ask Gemini to write the SQL

In [ ]:
SCHEMA = """
TABLE customers:
- customer_id INTEGER PRIMARY KEY
- customer_name TEXT
- city TEXT

TABLE products:
- product_id INTEGER PRIMARY KEY
- product_name TEXT
- category TEXT
- price REAL

TABLE orders:
- order_id INTEGER PRIMARY KEY
- customer_id INTEGER
- order_date TEXT (YYYY-MM-DD)
- status TEXT (completed, pending, cancelled)

TABLE order_items:
- item_id INTEGER PRIMARY KEY
- order_id INTEGER
- product_id INTEGER
- quantity INTEGER
- unit_price REAL

RELATIONSHIPS:
orders.customer_id = customers.customer_id
order_items.order_id = orders.order_id
order_items.product_id = products.product_id

BUSINESS RULE:
Revenue means SUM(order_items.quantity * order_items.unit_price)
for completed orders only, unless the user explicitly asks for
a different measure.

The database uses SQLite.
"""

SQL_PROMPT = """
You are a careful Text-to-SQL agent for an e-commerce database.

Your job is to translate the user's question into one valid SQLite query.

Database schema:
{schema}

Today's date is {today}. Use it for relative dates.

Rules:
1. Return only SQL, without Markdown fences or explanations.
2. Generate only SELECT or WITH queries.
3. Never modify the database.
4. Use only tables and columns in the provided schema.
5. Use correct joins and aggregate functions.
6. For revenue, use quantity * unit_price and completed orders,
   unless the question specifies otherwise.
7. For relative dates such as last Friday, use SQLite date functions.
8. If an earlier query failed, use the error message to correct it.
9. Never invent missing data.
10. Use LIMIT when returning a list of results.

User question:
{question}

Previous failed SQL, if any:
{previous_sql}

Error message, if any:
{error_message}

Return the corrected SQL query only.
"""


def generate_sql(question, previous_sql="", error_message=""):
    prompt = SQL_PROMPT.format(
        schema=SCHEMA,
        today=date.today().isoformat(),
        question=question,
        previous_sql=previous_sql or "None",
        error_message=error_message or "None",
    )
    sql = get_text(llm.invoke(prompt).content).strip()
    sql = re.sub(r"^```(?:sql)?\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\s*```$", "", sql)
    return sql.strip()

## Build the agent with LangGraph

In [ ]:
class AgentState(TypedDict, total=False):
    question: str
    sql: str
    error: str
    result: Optional[pd.DataFrame]
    attempts: int
    success: bool
    log: list
    output_type: str
    answer: str


def write_sql(state):
    sql = generate_sql(state["question"], previous_sql=state.get("sql", ""), error_message=state.get("error", ""))
    attempts = state.get("attempts", 0) + 1
    print(f"\nAttempt {attempts}")
    print("Generated SQL:")
    print(sql)
    return {"sql": sql, "attempts": attempts}


def run_sql(state):
    log = list(state.get("log", []))
    try:
        result = execute_sql(state["sql"])
        print("Query executed successfully.")
        log.append({"attempt": state["attempts"], "sql": state["sql"], "error": None})
        return {"result": result, "error": "", "success": True, "log": log}
    except Exception as error:
        print("SQL error:", error)
        log.append({"attempt": state["attempts"], "sql": state["sql"], "error": str(error)})
        return {"result": None, "error": str(error), "success": False, "log": log}


def route_after_run(state):
    if state["success"]:
        return "done"
    if state["attempts"] >= MAX_RETRIES + 1:
        print("Maximum retries reached. The agent gives up without crashing.")
        return "give_up"
    print("Asking Gemini to correct the query...")
    return "retry"


def summarize_result(question, sql, result):
    result_text = "The query returned no rows." if result is None or result.empty else result.to_string(index=False)
    prompt = f"""
You are a business data analyst.

Original question:
{question}

SQL query:
{sql}

Database result:
{result_text}

Explain the result clearly in plain English.
Use the actual values from the database.
Do not invent facts or claim that the result proves something
that the data does not establish.
If the result is empty, explain that no matching records were found.
"""
    return get_text(llm.invoke(prompt).content)


def choose_output_type(question):
    prompt = f"""
Decide how to present the result of this database question.

Question: {question}

Return exactly one word:
CHART - if the user explicitly asks for a chart, graph,
plot, or visual comparison.
TEXT - otherwise.
"""
    choice = get_text(llm.invoke(prompt).content).strip().upper()
    return "CHART" if "CHART" in choice else "TEXT"


def create_chart(result, question, save_as="agent_chart.png"):
    if result is None or result.empty or len(result.columns) < 2:
        print("Not enough data to draw a chart.")
        return
    x = result.iloc[:, 0].astype(str)
    y = pd.to_numeric(result.iloc[:, 1], errors="coerce")
    valid = y.notna()
    x, y = x[valid], y[valid]
    if len(y) == 0:
        print("No numeric values available to chart.")
        return
    plt.figure(figsize=(10, 5))
    plt.bar(x, y)
    plt.title(question)
    plt.xlabel(result.columns[0])
    plt.ylabel(result.columns[1])
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(save_as, dpi=150, bbox_inches="tight")   # save BEFORE show
    plt.show()


def present(state):
    question, result = state["question"], state["result"]
    print("\nResult:")
    display(result)
    output_type = choose_output_type(question)
    if output_type == "CHART":
        create_chart(result, question)
        answer = "(chart shown)"
    else:
        answer = summarize_result(question, state["sql"], result)
        print("\nBusiness answer:")
        print(answer)
    return {"output_type": output_type, "answer": answer}


graph = StateGraph(AgentState)
graph.add_node("write_sql", write_sql)
graph.add_node("run_sql", run_sql)
graph.add_node("present", present)

graph.add_edge(START, "write_sql")
graph.add_edge("write_sql", "run_sql")
graph.add_conditional_edges("run_sql", route_after_run,
                            {"retry": "write_sql", "done": "present", "give_up": END})
graph.add_edge("present", END)

sql_agent = graph.compile()


def ask_agent(question):
    print("=" * 70)
    print("QUESTION:", question)
    state = sql_agent.invoke({"question": question, "attempts": 0, "log": []})
    if not state.get("success"):
        print(f"\nThe agent could not answer after {state['attempts']} attempts. Last error: {state['error']}")
    return state


print("LangGraph agent is ready.")

## Test the agent on normal questions

In [ ]:
test_questions = [
    "What is the total revenue from completed orders?",
    "Which five products have the highest sales revenue?",
    "Which customers have placed the most orders?",
    "How much revenue did we make last Friday?",
    "How much revenue did we make in the last 7 days?",
    "How many orders are pending?",
]

summary = []
for q in test_questions:
    state = ask_agent(q)
    summary.append({"question": q, "attempts": state["attempts"], "success": state["success"]})

print("\n" + "=" * 70)
display(pd.DataFrame(summary))

## Prove that self-correction works

In [ ]:
real_generate_sql = generate_sql
calls = {"n": 0}

def broken_first_try(question, previous_sql="", error_message=""):
    calls["n"] += 1
    if calls["n"] == 1:
        return "SELECT SUM(price * qty) FROM orders_items"      # wrong table and columns, on purpose
    return real_generate_sql(question, previous_sql=previous_sql, error_message=error_message)

generate_sql = broken_first_try
state = ask_agent("What is the total revenue from completed orders?")
generate_sql = real_generate_sql                                # put the real function back

print("\nAttempt log:")
display(pd.DataFrame(state["log"]))

## Prove that the agent does not crash when it cannot recover

In [ ]:
real_generate_sql = generate_sql

def always_broken(question, previous_sql="", error_message=""):
    return "SELECT * FROM table_that_does_not_exist"

generate_sql = always_broken
state = ask_agent("What is the total revenue from completed orders?")
generate_sql = real_generate_sql

print("success:", state["success"], "| attempts:", state["attempts"])

## (Bonus): The chart tool


In [ ]:
state = ask_agent("Show a bar chart of the revenue for each product category.")

## Step 10 (Bonus): Read-only protection


In [ ]:
def table_counts():
    with sqlite3.connect(readonly_uri, uri=True) as c:
        return {t: c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                for t in ["customers", "products", "orders", "order_items"]}

before = table_counts()

security_tests = [
    "DELETE FROM customers",
    "DROP TABLE products",
    "UPDATE products SET price = 0",
    "INSERT INTO customers VALUES (99, 'Test', 'Cairo')",
    "SELECT * FROM customers; DROP TABLE orders",
    "WITH x AS (SELECT 1) DELETE FROM orders",
]

for query in security_tests:
    try:
        execute_sql(query)
        print("UNEXPECTED: query was allowed:", query)
    except Exception as error:
        print("BLOCKED:", query)
        print("   reason:", error)

after = table_counts()
print("\nRow counts before:", before)
print("Row counts after :", after)
print("Data unchanged:", before == after)